# 00 · Glucose 101 and the data

**Goal:** understand what we're modelling, look at the data the way a signal-processing person
would, and measure two things every later notebook depends on: the noise floor and the effect of
insulin.

**Time:** ~45 min · **You'll write:** `time_in_range`, `noise_floor`, `event_triggered_average`

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

## 1. The system

In type-1 diabetes the pancreas makes no insulin. The person:
- measures glucose with a **CGM** (continuous glucose monitor) every 5 minutes;
- delivers insulin through a pump: a constant **basal** rate plus **boluses** at meals;
- eats carbohydrates, which raise glucose after a delay.

| Biology | Signal-processing view |
|---|---|
| glucose (mg/dL) | output $y[n]$, sampled at 5 min |
| CGM | noisy measurement of $y$ (the noise is coloured, not white) |
| insulin, carbs | two known inputs $u[n]$, $r[n]$ |
| absorption from skin and gut | low-pass filters: a few first-order lags in series |
| insulin sensitivity | a gain that differs several-fold between people |
| target range 70–180 mg/dL | below 70 (hypoglycaemia) is dangerous within minutes |

Insulin **lowers** glucose and carbs **raise** it. Both act with delays of tens of minutes to
hours, which is exactly why forecasting is useful.

## 2. The data

We use 30 virtual patients from the UVA/Padova simulator (`simglucose`, MIT license), 10 days
each. `data/cohort.npz` holds arrays of shape `[patients, time]` on a 5-minute grid.

In [ ]:
from biotwin.data.dataset import Cohort, STEPS_PER_DAY

cohort = Cohort.load("data/cohort.npz")
print("patients:", list(cohort.names[:3]), "...", cohort.cgm.shape)
for k in ["cgm", "bg", "insulin", "carbs"]:
    print(f"{k:8s} {getattr(cohort, k).shape}  e.g. {getattr(cohort, k)[0, 100:104].round(2)}")
print("basal (U per 5 min):", cohort.basal[:3].round(4))

- `cgm`: what the sensor reports, i.e. what a model is allowed to see.
- `bg`: the simulator's noise-free glucose at the sensor site. **Never a model input.** We use
  it only to measure the sensor noise.
- `insulin`: units delivered in each 5-minute step (basal + boluses).
- `carbs`: grams eaten in each step.

In [ ]:
def plot_day(name, day):
    p = cohort.idx(name)
    sl = slice(day * STEPS_PER_DAY, (day + 1) * STEPS_PER_DAY)
    hrs = np.arange(STEPS_PER_DAY) / 12
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    a1.plot(hrs, cohort.bg[p, sl], lw=1, label="noise-free glucose")
    a1.plot(hrs, cohort.cgm[p, sl], ".", ms=2, label="CGM")
    a1.axhspan(70, 180, color="g", alpha=0.07)
    a1.set_ylabel("mg/dL"); a1.set_title(f"{name}, day {day}"); a1.legend()
    a2.bar(hrs, cohort.carbs[p, sl], width=0.08, color="tab:orange", label="carbs (g)")
    a2.bar(hrs, -10 * (cohort.insulin[p, sl] - cohort.basal[p]), width=0.08, color="tab:purple", label="bolus ×10 (U)")
    a2.set_xlabel("hour"); a2.legend(); plt.tight_layout(); plt.show()

plot_day("adult#003", 2)
plot_day("child#005", 2)

## Exercise 1 — time in range

Clinicians summarise a CGM trace by its **time in range (TIR)**: the fraction of readings with
70 ≤ glucose ≤ 180 (both ends included).


<details><summary><b>Hint</b></summary>

A boolean mask, then `.mean()`. `np.asarray` makes it work for lists too.

</details>

In [ ]:
def time_in_range(cgm, lo=70.0, hi=180.0):
    # TODO: return a Python float in [0, 1]
    raise NotImplementedError

def check(f):
    close(f(np.array([60, 70, 100, 180, 181, 250])), 0.5, what="TIR")
    assert isinstance(f([100.0]), float), "return a Python float"

time_in_range = use(time_in_range, sol.time_in_range, check)

In [ ]:
tir = {str(nm): time_in_range(cohort.cgm[i]) for i, nm in enumerate(cohort.names)}
groups = sorted({k.split("#")[0] for k in tir})
plt.figure(figsize=(10, 3))
for g in groups:
    vals = [v for k, v in tir.items() if k.startswith(g)]
    plt.scatter([g] * len(vals), vals, alpha=0.7)
plt.ylabel("time in range"); plt.title("each dot is one patient"); plt.show()

## Exercise 2 — the noise floor

How far is CGM from the noise-free glucose? Compute the RMSE between them. **No forecaster can
reliably beat this number against CGM targets**, so remember it.


<details><summary><b>Hint</b></summary>

$\sqrt{\frac{1}{N}\sum (\text{cgm}-\text{bg})^2}$. It works on arrays of any shape.

</details>

In [ ]:
def noise_floor(cgm, bg):
    # TODO: RMSE between the two arrays, as a float
    raise NotImplementedError

def check(f):
    close(f([1.0, 2.0, 3.0], [1.0, 2.0, 5.0]), np.sqrt(4 / 3), what="RMSE")

noise_floor = use(noise_floor, sol.noise_floor, check)
print(f"sensor noise floor: {noise_floor(cohort.cgm, cohort.bg):.1f} mg/dL")

The CGM error isn't white noise. Below is its autocorrelation: it stays correlated for tens of
minutes, so a slow wobble on the CGM can look like a real glucose change.

In [ ]:
e = (cohort.cgm - cohort.bg)[:, :2000]
e = e - e.mean(1, keepdims=True)
lags = np.arange(0, 25)
ac = [np.mean(e[:, : e.shape[1] - L] * e[:, L:]) / np.mean(e * e) for L in lags]
plt.stem(lags * 5, ac); plt.xlabel("lag (min)"); plt.ylabel("autocorrelation"); plt.title("CGM error"); plt.show()

## Exercise 3 — what does a meal do, and what does insulin do?

The classic SP tool here is the **event-triggered average** (in neuroscience it's called the
spike-triggered average). Cut a segment of the signal around each event and average the
segments. The noise and the unrelated activity average out, and the typical response remains.

Write `event_triggered_average(x, events, pre, post)` that returns the mean of `x[e-pre : e+post]`
over all events `e`, **skipping events whose segment would fall off either end of `x`**.


<details><summary><b>Hint</b></summary>

A list comprehension with a condition, then `np.mean(segments, axis=0)`. The result has length `pre + post`.

</details>

In [ ]:
def event_triggered_average(x, events, pre, post):
    # TODO
    raise NotImplementedError

def check(f):
    x = np.arange(100.0)
    close(f(x, [10, 50, 98], 2, 3), np.arange(28.0, 33.0), what="ETA")  # event 98 is skipped

event_triggered_average = use(event_triggered_average, sol.event_triggered_average, check)

Now apply it to the data. We find meal onsets (at least 20 g) and split them into meals **with**
a bolus nearby and meals **without** one: the 10 % that patients forgot. This is a natural
experiment, the closest thing to an intervention study we can do with observational data.

In [ ]:
pre, post = 12, 48  # 1 h before, 4 h after
curves = {"meal + bolus": [], "meal, no bolus": []}
for p in range(len(cohort.names)):
    c, dev = cohort.carbs[p], cohort.insulin[p] - cohort.basal[p]
    onsets = [t for t in range(pre, len(c) - post) if c[t] > 0 and c[t - 1] == 0 and c[t:t + 6].sum() >= 20]
    for t in onsets:
        near = dev[max(0, t - 3):t + 9].max()
        key = "meal + bolus" if near > 0.3 else ("meal, no bolus" if near < 0.05 else None)
        if key:
            seg = event_triggered_average(cohort.cgm[p], [t], pre, post)
            curves[key].append(seg - seg[pre])

mins = (np.arange(pre + post) - pre) * 5
for k, v in curves.items():
    plt.plot(mins, np.mean(v, 0), label=f"{k} (n={len(v)})")
plt.axvline(0, color="k", lw=0.8); plt.xlabel("minutes from meal"); plt.ylabel("Δ glucose (mg/dL)")
plt.legend(); plt.title("event-triggered average of CGM"); plt.show()

Read the plot as an SP person:
- The gap between the two curves is the **insulin effect**. It needs about 45–60 min to show up
  (subcutaneous absorption plus action delay) and lasts for hours: a slow low-pass response.
- Without insulin, glucose stays high for hours. The body's own glucose clearance is slow.
- These are averages over different people, so they mix fast and slow responders. Section 9 of
  the project (personalisation) is about un-mixing that.

## Quiz

Answer in your own words before opening the answers.

**1. Why do we keep the noise-free glucose `bg` out of every model input, even though it exists in the data?**

<details><summary>Answer</summary>

A real system only has the CGM. Using `bg` as an input would be leakage from a signal that wouldn't exist at deployment. We use it only for analysis, for example to measure the noise floor.

</details>

**2. Insulin passes through roughly a third-order low-pass filter before it acts. What does that predict about the glucose response to a bolus?**

<details><summary>Answer</summary>

A delayed, smooth response. Almost nothing for the first ~20–30 min, then a slow drop that peaks one to two hours later and lasts for several hours. No step-like jumps. That's what the event-triggered average shows.

</details>

**3. The simulator's patients sometimes skip boluses and miscount carbs. Why did we add that on purpose?**

<details><summary>Answer</summary>

If every meal always came with a perfectly matched bolus, insulin and carbs would be collinear inputs and no estimator could separate their effects (identifiability). The imperfections give independent variation, like a persistently exciting input in system identification.

</details>

**4. A model reaches 11 mg/dL RMSE against CGM at a 5-min horizon. Is that impressive?**

<details><summary>Answer</summary>

Not really. The noise floor is ~11.5, and over 5 minutes glucose barely moves, so persistence is close too. Compare against the noise floor and against persistence at the same horizon.

</details>

**5. The CGM error is autocorrelated over tens of minutes. Why does that matter for forecasting and for uncertainty?**

<details><summary>Answer</summary>

A forecaster can partly 'predict the noise', and interval methods that assume independent errors are miscalibrated. Errors on adjacent windows are also correlated, so N windows hold fewer than N independent pieces of evidence (this comes back in notebook 06).

</details>